# Q2 – Tuples, Mutability and Copying
**CSA0812 | Darshan A | Register Number: 1973260011 | 7 Marks**

## Aim
To investigate tuples, mutability, aliasing, shallow copying and deep copying. The task also demonstrates independent nested lists, tuple dictionary keys, memory use and performance.

## Prediction
The repeated-row grid will share one inner list, so changing one row affects all rows. A list comprehension will create independent rows. A deep copy will duplicate nested objects, unlike a shallow copy. Tuples can be dictionary keys when their contents are hashable; lists cannot.

## Pseudocode
1. Create an incorrectly repeated grid and update one cell.
2. Create a corrected grid using a list comprehension and update one cell.
3. Compare row identities using `is` and `id()`.
4. Demonstrate aliasing, shallow copying and deep copying.
5. Demonstrate tuple unpacking, tuple dictionary keys and a mutable list inside a tuple.
6. Compare memory and timing for a list and tuple of one million values.
7. Demonstrate a sparse dictionary grid and analyse its space use.

In [ ]:
# 1. Required TODO program: shared rows vs independent rows
rows, cols = 3, 4

bad = [[0] * cols] * rows
bad[0][1] = 7

# TODO 1 completed: each row is a separate list
good = [[0] * cols for _ in range(rows)]
good[0][1] = 7

print("bad :", bad)
print("good:", good)
print("Same bad rows?", bad[0] is bad[1])
print("Same good rows?", good[0] is good[1])

# Tuples as dictionary keys
stops = {
    (13.0827, 80.2707): "Chennai Central",
    (13.0418, 80.2341): "T. Nagar"
}
print("Stop lookup:", stops[(13.0418, 80.2341)])

# A list cannot be used as a dictionary key
try:
    stops[[13.0418, 80.2341]] = "T. Nagar"
except TypeError as error:
    print("List-key error:", error)

bad : [[0, 7, 0, 0], [0, 7, 0, 0], [0, 7, 0, 0]]
good: [[0, 7, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
Same bad rows? True
Same good rows? False
Stop lookup: T. Nagar
List-key error: unhashable type: 'list'


## 2. Aliasing, shallow copy, deep copy, unpacking and mutability

In [ ]:
import copy

original = [[10, 20], [30, 40]]
alias = original
shallow = copy.copy(original)
slice_copy = original[:]
deep = copy.deepcopy(original)

print("Identity checks:")
print("original is alias:", original is alias)
print("original is shallow:", original is shallow)
print("original is deep:", original is deep)

original[0][0] = 999
print("\nAfter changing original[0][0]:")
print("original:", original)
print("alias:", alias)
print("shallow:", shallow)
print("slice copy:", slice_copy)
print("deep:", deep)

# Tuple unpacking
stops_list = [
    ("Chennai Central", 13.0827, 80.2707),
    ("T. Nagar", 13.0418, 80.2341),
    ("Guindy", 13.0067, 80.2206)
]
print("\nTuple unpacking:")
for name, latitude, longitude in stops_list:
    print(name, latitude, longitude)

# A tuple is immutable, but a list stored inside it remains mutable
record = ([10, 20, 30], "Passenger data")
record[0].append(40)
print("\nTuple containing a list after append:", record)

# A function mutating its list argument changes the same list object
def add_passenger(passengers):
    passengers.append("New Passenger")

bus_passengers = ["Ravi", "Priya"]
add_passenger(bus_passengers)
print("List after function call:", bus_passengers)

Identity checks:
original is alias: True
original is shallow: False
original is deep: False

After changing original[0][0]:
original: [[999, 20], [30, 40]]
alias: [[999, 20], [30, 40]]
shallow: [[999, 20], [30, 40]]
slice copy: [[999, 20], [30, 40]]
deep: [[10, 20], [30, 40]]

Tuple unpacking:
Chennai Central 13.0827 80.2707
T. Nagar 13.0418 80.2341
Guindy 13.0067 80.2206

Tuple containing a list after append: ([10, 20, 30, 40], 'Passenger data')
List after function call: ['Ravi', 'Priya', 'New Passenger']


## 3. Memory and performance investigation
`sys.getsizeof()` reports the shallow size of the container. The second measurement adds the sizes of the contained integer objects. Timing results vary by computer and execution session; use the values printed by your own run.

In [ ]:
import sys
import time
import statistics

size = 1_000_000

# Memory comparison
values_list = list(range(size))
values_tuple = tuple(range(size))

list_shallow = sys.getsizeof(values_list)
tuple_shallow = sys.getsizeof(values_tuple)
list_with_elements = list_shallow + sum(sys.getsizeof(x) for x in values_list)
tuple_with_elements = tuple_shallow + sum(sys.getsizeof(x) for x in values_tuple)

print("ELEMENTS:", size)
print("List container:", list_shallow, "bytes")
print("Tuple container:", tuple_shallow, "bytes")
print("List plus element sizes:", list_with_elements, "bytes")
print("Tuple plus element sizes:", tuple_with_elements, "bytes")

# Creation and indexing timing; repeat and report the median
def median_time(function, repeats=5):
    measurements = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = function()
        measurements.append(time.perf_counter() - start)
    return statistics.median(measurements), result

list_create_time, timed_list = median_time(lambda: list(range(size)))
tuple_create_time, timed_tuple = median_time(lambda: tuple(range(size)))

indices = range(0, size, 10)  # 100,000 index operations
def index_list():
    total = 0
    for i in indices:
        total += timed_list[i]
    return total

def index_tuple():
    total = 0
    for i in indices:
        total += timed_tuple[i]
    return total

list_index_time, list_total = median_time(index_list)
tuple_index_time, tuple_total = median_time(index_tuple)

print("\nMEDIAN OF 5 RUNS")
print("List creation:", round(list_create_time, 6), "seconds")
print("Tuple creation:", round(tuple_create_time, 6), "seconds")
print("List indexing (100,000 accesses):", round(list_index_time, 6), "seconds")
print("Tuple indexing (100,000 accesses):", round(tuple_index_time, 6), "seconds")
print("Indexing totals match:", list_total == tuple_total)

ELEMENTS: 1000000
List container: 8000056 bytes
Tuple container: 8000040 bytes
List plus element sizes: 36000056 bytes
Tuple plus element sizes: 36000040 bytes

MEDIAN OF 5 RUNS
List creation: 0.034991 seconds
Tuple creation: 0.036163 seconds
List indexing (100,000 accesses): 0.008944 seconds
Tuple indexing (100,000 accesses): 0.008942 seconds
Indexing totals match: True


## 4. Sparse dictionary grid alternative

In [ ]:
rows, cols = 3, 4
zones = {}

# Store only non-empty zones, using (row, column) tuple keys
zones[(0, 1)] = 7
zones[(2, 3)] = 5

print("Passengers in zone (0, 1):", zones.get((0, 1), 0))
print("Passengers in zone (1, 1):", zones.get((1, 1), 0))

zones[(1, 2)] = 3
print("Non-empty zones:")
for position, passengers in zones.items():
    print(position, "->", passengers)
print("Stored non-empty zones:", len(zones))

Passengers in zone (0, 1): 7
Passengers in zone (1, 1): 0
Non-empty zones:
(0, 1) -> 7
(2, 3) -> 5
(1, 2) -> 3
Stored non-empty zones: 3


## 5. Written analysis and complexity

### Why tuples can be dictionary keys but lists cannot
Tuples are immutable, and tuples containing hashable elements can be hashed. Lists are mutable and unhashable, so Python rejects them as dictionary keys.

### Memory diagrams
Incorrect `[[0] * 3] * 3`:
```text
bad ──┬──> Row A
      ├──> Row A
      └──> Row A
```
All outer positions refer to the same inner list.

Correct `[[0] * 3 for _ in range(3)]`:
```text
good ──┬──> Row A
       ├──> Row B
       └──> Row C
```
Each row is a separate list.

### Shallow copy and deep copy
A shallow copy creates a new outer container but shares nested objects. It is suitable when elements are immutable or nested sharing is intended. A deep copy recursively copies nested objects and is appropriate when nested mutable data must be independent.

### Function arguments
A function receives a reference to the same list object. Mutating that object inside the function is therefore visible to the caller.

### Tuple containing a list
A tuple cannot have an element replaced, but a list stored inside it remains mutable. Calling `append()` on that inner list succeeds.

### Complexity
| Operation/design | Time | Space |
|---|---:|---:|
| List/tuple indexing | O(1) | O(n) storage |
| Dictionary lookup | O(1) average | O(n) stored entries |
| Shallow copy | O(n) | O(n) new outer container |
| Deep copy (this nested example) | O(n) | O(n) copied objects |
| Full grid | O(1) average cell access | O(rows × columns) |
| Sparse dictionary grid | O(1) average lookup | O(non-empty zones) |

### Conclusion
Tuples are suitable for fixed records such as GPS coordinates. Lists are appropriate when data must change. A list comprehension avoids shared-row bugs. A sparse dictionary is useful when most grid positions are empty; its access is O(1) average and its storage depends on the number of non-empty zones.

**Submission note:** The assignment requires individual execution. Run every code cell in Google Colab and retain your own outputs and screenshots. The timing and memory values are generated when the notebook is executed.